# RFM 评分方法、权重方案与 I 指标集成实验

从主分析流程中摘出的三个实验，用于确定：
1. **评分方法**（实验一）：qcut等频分箱 vs Min-Max归一化 vs Z-Score
2. **权重方案**（实验二）：等权 vs 偏货币 vs 偏活跃
3. **I 指标集成方式**（实验三）：加法 vs 乘法温和 vs 乘法激进

In [ ]:
# 库导入与全局配置\nimport os, glob\n\nimport pandas as pd\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport matplotlib.font_manager as fm\nimport seaborn as sns\nfrom scipy.stats import gaussian_kde\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# ===== Seaborn 样式（必须在字体设置之前！） =====\nsns.set_style('whitegrid')\n\n# ===== 中文字体设置（必须在 sns.set_style 之后，否则会被覆盖） =====\nfm._load_fontmanager(try_read_cache=False)\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']\nplt.rcParams['font.family'] = 'sans-serif'\nplt.rcParams['axes.unicode_minus'] = False\n\nfrom matplotlib.ft2font import FT2Font\nfont = FT2Font('C:/Windows/Fonts/simhei.ttf')\ntest_glyph = font.get_char_index(ord('用'))\nprint(f"SimHei 字体: {'✅ 正常' if test_glyph > 0 else '❌ 异常'}")\nprint(f"font.sans-serif={plt.rcParams['font.sans-serif'][:3]}")

In [ ]:
# 数据加载\ninput_path = r"D:\project\电商用户价值分层\user_personalized_features.csv"\n\ndf = pd.read_csv(input_path, index_col=0)\nprint(f"数据规模: {df.shape[0]} 行 x {df.shape[1]} 列")\ndf.head(3)

In [ ]:
# 3.1 评分函数定义\n\ndef min_max_score(series, reverse=False):\n    # Min-Max 归一化到 0-100\n    mn, mx = series.min(), series.max()\n    if mx == mn:\n        return pd.Series(50.0, index=series.index)\n    if reverse:\n        return ((mx - series) / (mx - mn)) * 100\n    return ((series - mn) / (mx - mn)) * 100\n\ndef qcut_score(series, q=5, reverse=False):\n    # 等频分箱 1-5 分\n    if reverse:\n        return 6 - pd.qcut(series.rank(method='first'), q=q, labels=range(1, q+1)).astype(int)\n    return pd.qcut(series.rank(method='first'), q=q, labels=range(1, q+1)).astype(int)\n\nR_raw = df['Last_Login_Days_Ago']\nF_raw = df['Purchase_Frequency']\nM_raw = df['Total_Spending']\n\nprint("=" * 60)\nprint("实验一：评分方法对比")\nprint("=" * 60)\n\n# 方案A: qcut (1-5)\nr_q = qcut_score(R_raw, reverse=True); f_q = qcut_score(F_raw); m_q = qcut_score(M_raw)\n# 方案B: Min-Max (0-100)\nr_mm = min_max_score(R_raw, reverse=True); f_mm = min_max_score(F_raw); m_mm = min_max_score(M_raw)\n# 方案C: Z-Score\nr_z = (R_raw - R_raw.mean()) / R_raw.std() * (-1)\nf_z = (F_raw - F_raw.mean()) / F_raw.std()\nm_z = (M_raw - M_raw.mean()) / M_raw.std()\n\ncomparison = pd.DataFrame({\n    '评分方法': ['qcut(1-5)', 'Min-Max(0-100)', 'Z-Score'],\n    'R_范围': [f'{r_q.min()}-{r_q.max()}', f'{r_mm.min():.0f}-{r_mm.max():.0f}', f'{r_z.min():.2f}~{r_z.max():.2f}'],\n    'F_范围': [f'{f_q.min()}-{f_q.max()}', f'{f_mm.min():.0f}-{f_mm.max():.0f}', f'{f_z.min():.2f}~{f_z.max():.2f}'],\n    'M_范围': [f'{m_q.min()}-{m_q.max()}', f'{m_mm.min():.0f}-{m_mm.max():.0f}', f'{m_z.min():.2f}~{m_z.max():.2f}'],\n    '信息粒度': ['仅5档，粗', '连续100档，细', '连续，但有负值'],\n    '可解释性': ['直观', '直观', '需标准化知识'],\n    '可比性': ['差(序数不可比)', '好(连续可比)', '中(需统一标准化)'],\n})\ndisplay(comparison)\n\nprint("\n结论: qcut仅5档区分度低；Z-Score有负值不利于业务沟通。")\nprint("选择: Min-Max归一化(0-100)作为统一评分标准")

In [ ]:
# 3.2 实验二：RFM 权重方案对比\n# 统一使用 Min-Max 评分\n\nR = min_max_score(df['Last_Login_Days_Ago'], reverse=True)\nF = min_max_score(df['Purchase_Frequency'], reverse=False)\nM = min_max_score(df['Total_Spending'], reverse=False)\n\n# 三种权重\nschemes = {\n    'W1_等权重':     {'wR': 0.33, 'wF': 0.33, 'wM': 0.33},\n    'W2_偏货币':     {'wR': 0.20, 'wF': 0.30, 'wM': 0.50},\n    'W3_偏活跃':     {'wR': 0.40, 'wF': 0.30, 'wM': 0.30},\n}\n\nresults = []\nfor name, w in schemes.items():\n    score = w['wR']*R + w['wF']*F + w['wM']*M\n    tier = pd.qcut(score, q=5, labels=['L1_最低','L2_较低','L3_中等','L4_较高','L5_最高'])\n    grouped = df.groupby(tier)['Total_Spending'].mean()\n    results.append({\n        '方案': f'{name}\
(R={w["wR"]},F={w["wF"]},M={w["wM"]})',\n        'L1_最低消费': f'{grouped["L1_最低"]:.0f}',\n        'L5_最高消费': f'{grouped["L5_最高"]:.0f}',\n        'L5/L1比值': f'{grouped["L5_最高"]/grouped["L1_最低"]:.2f}x',\n        '得分均值': f'{score.mean():.1f}',\n        '得分标准差': f'{score.std():.1f}',\n        '与消费相关系数': f'{score.corr(df["Total_Spending"]):.3f}',\n    })\n\ndisplay(pd.DataFrame(results))\n\n# 可视化\nfig, axes = plt.subplots(1, 3, figsize=(16, 5))\nfor idx, (name, w) in enumerate(schemes.items()):\n    score = w['wR']*R + w['wF']*F + w['wM']*M\n    tier = pd.qcut(score, q=5, labels=['L1','L2','L3','L4','L5'])\n    grouped = df.groupby(tier)['Total_Spending'].mean()\n    bars = axes[idx].bar(grouped.index, grouped.values, color=sns.color_palette('Blues_d', 5))\n    axes[idx].set_title(f'{name}\
(R={w["wR"]},F={w["wF"]},M={w["wM"]})', fontsize=11, fontweight='bold')\n    axes[idx].set_ylabel('平均消费(元)')\n    for bar, v in zip(bars, grouped.values):\n        axes[idx].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, f'{v:.0f}', ha='center', fontsize=9)\n\nplt.suptitle('各权重方案的分层区分度对比', fontsize=14, fontweight='bold', y=1.02)\nplt.tight_layout()\nplt.show()\n\nprint("\n结论: W2偏货币方案 L5/L1比值最高，且与消费的相关系数最大，区分度最优。")\nprint("选择: W2 (R=0.2, F=0.3, M=0.5) 作为基线权重")

In [ ]:
# 5.2 实验三：I 指标集成方式对比

# 先构建 RFM 得分（基于实验一、实验二的结论）
R = min_max_score(df['Last_Login_Days_Ago'], reverse=True)
F = min_max_score(df['Purchase_Frequency'], reverse=False)
M = min_max_score(df['Total_Spending'], reverse=False)
df['RFM_Score'] = 0.2 * R + 0.3 * F + 0.5 * M

# 构建 I 指标
df['Time_Norm'] = min_max_score(df['Time_Spent_on_Site_Minutes'])
df['Pages_Norm'] = min_max_score(df['Pages_Viewed'])
df['I_Score'] = 0.5 * df['Time_Norm'] + 0.5 * df['Pages_Norm']

# 方案I1: 加法   Final = RFM + 0.2*I
# 方案I2: 乘法温和 Final = RFM * (1 + I/500)  -- I=100时放大20%
# 方案I3: 乘法激进 Final = RFM * (1 + I/250)  -- I=100时放大40%
df['Final_I1_Add'] = df['RFM_Score'] + 0.2 * df['I_Score']
df['Final_I2_Mild'] = df['RFM_Score'] * (1 + df['I_Score'] / 500)
df['Final_I3_Strong'] = df['RFM_Score'] * (1 + df['I_Score'] / 250)

# 评估: 能否有效提升"高意向低RFM"用户的得分?
hi_i_lo_rfm = df[(df['I_Score'] > df['I_Score'].quantile(0.75)) &
                  (df['RFM_Score'] < df['RFM_Score'].quantile(0.25))]

print("=" * 60)
print("实验三：I指标集成方式对比")
print("=" * 60)
print(f"\n[高意向低RFM]目标用户: {len(hi_i_lo_rfm)}人")
print(f"  原始RFM均值: {hi_i_lo_rfm['RFM_Score'].mean():.1f} | I均值: {hi_i_lo_rfm['I_Score'].mean():.1f}")
print(f"  加法方案  最终得分: {hi_i_lo_rfm['Final_I1_Add'].mean():.1f}")
print(f"  乘法和方案 最终得分: {hi_i_lo_rfm['Final_I2_Mild'].mean():.1f}")
print(f"  乘法强方案 最终得分: {hi_i_lo_rfm['Final_I3_Strong'].mean():.1f}")

# 与消费的相关性
for col, name in [('Final_I1_Add','加法'),('Final_I2_Mild','乘法和'),('Final_I3_Strong','乘法强')]:
    print(f"  {name}: 与Total_Spending相关系数 = {df[col].corr(df['Total_Spending']):.3f}")

# 分布可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, (col, title) in enumerate([
    ('Final_I1_Add', '加法 (RFM+0.2xI)'),
    ('Final_I2_Mild', '乘法和 (RFMx(1+I/500))'),
    ('Final_I3_Strong', '乘法强 (RFMx(1+I/250))')
]):
    axes[idx].hist(df[col], bins=40, color='#0B5CAD', edgecolor='white', alpha=0.8)
    axes[idx].axvline(df[col].mean(), color='red', ls='--', lw=1.5,
                      label=f'均值:{df[col].mean():.1f}')
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].legend()
plt.suptitle('三种 I 集成方案的得分分布', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n结论: 乘法和方案既保持了RFM主体结构，又能有效放大高意向用户得分。")
print("选择: I2 乘法温和方案")